In [1]:
import zipfile
import os
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
import optuna # <-- NUEVO: Importar optuna
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
zip_file_path = '../data/MeIA2025-Reto-01.zip'

extracted_folder_path = '../data/extracted_corpus/'

# 2. Crear la carpeta de extracción si no existe
if not os.path.exists(extracted_folder_path):
    os.makedirs(extracted_folder_path)
    print(f"Carpeta '{extracted_folder_path}' creada.")
# 3. Descomprimir archivo  
try:
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_folder_path)
    print(f"'{zip_file_path}' descomprimido exitosamente en '{extracted_folder_path}'.")
except FileNotFoundError:
    print(f"Error: El archivo ZIP no se encontró en '{zip_file_path}'. Verifica la ruta.")
except Exception as e:
    print(f"Ocurrió un error al descomprimir el archivo: {e}")
    
# 4. Listar los archivos descomprimidos (para verificar)
print("\nArchivos en la carpeta del corpus:")
corpus_files = os.listdir(extracted_folder_path)
for file_name in corpus_files:
    print(f"- {file_name}")

# Definir la ruta base donde se extrajo el contenido del ZIP
# Asegúrate de que esta ruta sea correcta relativa a tu notebook test.ipynb
# Si tu notebook está en 'notebooks/' y la extracción está en 'data/extracted_corpus/Datos-MelA-Reto-01/'
base_extracted_path = '../data/extracted_corpus/Datos-MeIA-Reto-01/'

# Rutas completas a los archivos XLSX
train_file_path = os.path.join(base_extracted_path, 'MeIA_2025_train.xlsx')
test_file_path = os.path.join(base_extracted_path, 'MeIA_2025_test_wo_labels.xlsx')

print(f"Intentando cargar el archivo de entrenamiento desde: {train_file_path}")
print(f"Intentando cargar el archivo de prueba desde: {test_file_path}")

try:
    # Cargar el dataset de entrenamiento
    
    df_train = pd.read_excel(train_file_path)
    print(f"Datos de entrenamiento cargados correctamente")
    # Cargar el dataset de prueba (sin etiquetas)
    df_test = pd.read_excel(test_file_path)
    print(f"Datos de test cargados correctamente")


except FileNotFoundError:
    print(f"Error: Uno de los archivos XLSX no se encontró.")
    print(f"Asegúrate de que las rutas sean correctas: '{train_file_path}' y '{test_file_path}'")
    print(f"Y que la carpeta 'Datos-MelA-Reto-01' esté dentro de 'extracted_corpus'.")
except Exception as e:
    print(f"Ocurrió un error al cargar los archivos Excel: {e}")

2025-06-17 20:00:51.001809: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750201251.752043  127152 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750201251.992512  127152 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750201254.207037  127152 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750201254.207436  127152 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1750201254.207438  127152 computation_placer.cc:177] computation placer alr

'../data/MeIA2025-Reto-01.zip' descomprimido exitosamente en '../data/extracted_corpus/'.

Archivos en la carpeta del corpus:
- Datos-MeIA-Reto-01
Intentando cargar el archivo de entrenamiento desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_train.xlsx
Intentando cargar el archivo de prueba desde: ../data/extracted_corpus/Datos-MeIA-Reto-01/MeIA_2025_test_wo_labels.xlsx
Datos de entrenamiento cargados correctamente
Datos de test cargados correctamente


In [3]:
from scipy.special import softmax
# --- 2. Preprocesamiento del Texto (¡Debe ser IDÉNTICO al de entrenamiento!) ---
print("Preprocesando texto de test...")
df_test['text'] = df_test.apply(
    lambda row: f"tipo: {str(row['Type']).lower()}. pueblo: {str(row['Town']).lower()}. reseña: {str(row['Review']).lower()}",
    axis=1
)
test_texts = df_test['text'].tolist()


# --- 3. Cargar Modelos y Realizar Predicciones en Bucle ---
model_paths = [f"./modelos_tabu/kfold_model_fold_{i+1}" for i in range(5)]
all_probabilities = []

# Cargamos el tokenizador (es el mismo para todos los modelos)
tokenizer = AutoTokenizer.from_pretrained(model_paths[0])

# Tokenizamos los datos de test una sola vez
tokenized_test_dataset = tokenizer(test_texts, padding="max_length", truncation=True, max_length=512, return_tensors="pt")
test_dataset = Dataset.from_dict(tokenized_test_dataset)

print(f"\nIniciando predicción con ensamble de 5 modelos...")
for i, path in enumerate(model_paths):
    print(f"--- Usando modelo del Fold {i+1}/5 desde '{path}' ---")
    
    # Cargar el modelo del fold actual
    model = AutoModelForSequenceClassification.from_pretrained(path)
    # Si tienes GPU, esto acelera enormemente el proceso
    if torch.cuda.is_available():
        model.to("cuda")

    # Creamos un Trainer simple solo para la predicción
    trainer = Trainer(model=model)
    
    # Obtenemos los logits (salidas crudas del modelo)
    raw_predictions = trainer.predict(test_dataset)
    
    # Convertimos los logits a probabilidades usando la función softmax
    # Esto nos da la "confianza" del modelo en cada una de las 5 clases
    probabilities = softmax(raw_predictions.predictions, axis=1)
    all_probabilities.append(probabilities)
    
    # Liberar memoria (opcional pero buena práctica)
    del model
    del trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# --- 4. Promediar Probabilidades y Obtener Predicción Final ---
print("\nPromediando las predicciones de todos los modelos...")
# Hacemos la media de las probabilidades obtenidas de los 5 modelos
# El resultado es una matriz de [2500 reseñas x 5 clases]
average_probabilities = np.mean(all_probabilities, axis=0)

# La predicción final para cada reseña es la clase (0 a 4) con la probabilidad promedio más alta
final_predictions_indices = np.argmax(average_probabilities, axis=1)

# Convertimos los índices (0-4) a las etiquetas del reto (1-5)
final_labels = final_predictions_indices + 1


# --- 5. Crear y Guardar el Archivo de Entrega ---
submission_df = pd.DataFrame({'ID': df_test['ID'], 'Polarity': final_labels})

Preprocesando texto de test...

Iniciando predicción con ensamble de 5 modelos...
--- Usando modelo del Fold 1/5 desde './modelos_tabu/kfold_model_fold_1' ---


--- Usando modelo del Fold 2/5 desde './modelos_tabu/kfold_model_fold_2' ---


--- Usando modelo del Fold 3/5 desde './modelos_tabu/kfold_model_fold_3' ---


--- Usando modelo del Fold 4/5 desde './modelos_tabu/kfold_model_fold_4' ---


--- Usando modelo del Fold 5/5 desde './modelos_tabu/kfold_model_fold_5' ---



Promediando las predicciones de todos los modelos...
